# SHL Grammar Scoring Engine

This project predicts grammar scores from spoken English audio for the SHL Research Intern assessment.

The approach combines handcrafted acoustic features with Whisper embeddings and uses an ensemble of CatBoost and ExtraTrees regressors.

- 91 handcrafted acoustic features
- 768-dimensional Whisper-small.en encoder embeddings
- CatBoost + ExtraTrees ensemble
- Local validation-based calibration
- Continuous grammar score from 0 to 5

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
from tqdm.auto import tqdm
from transformers import pipeline
from sklearn.ensemble import ExtraTreesRegressor
from catboost import CatBoostRegressor

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
base = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
train = pd.read_csv(f'{base}/train.csv')
test = pd.read_csv(f'{base}/test.csv')
sample = pd.read_csv(f'{base}/sample_submission.csv')

train_audio = f'{base}/train'
test_audio = f'{base}/test'
y_train = train['label'].values.astype(np.float32)

print('Train:', train.shape)
print('Test :', test.shape)
print('Labels:', sorted(train['label'].unique()))

## 1. Acoustic features

In [ ]:
def extract_features(path):
    y, sr = librosa.load(path, sr=16000, mono=True)
    feats = []

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    feats.extend(np.mean(mfcc, axis=1))
    feats.extend(np.std(mfcc, axis=1))

    delta = librosa.feature.delta(mfcc)
    feats.extend(np.mean(delta, axis=1))
    feats.extend(np.std(delta, axis=1))

    sc = librosa.feature.spectral_centroid(y=y, sr=sr)
    sb = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    sr_ = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)

    for f in [sc, sb, sr_, zcr, rms]:
        feats.append(np.mean(f))
        feats.append(np.std(f))

    feats.append(len(y) / sr)
    return np.array(feats, dtype=np.float32)

X_train = np.vstack([
    extract_features(os.path.join(train_audio, fn))
    for fn in tqdm(train['filename'], desc='Acoustic train')
])

X_test = np.vstack([
    extract_features(os.path.join(test_audio, fn))
    for fn in tqdm(test['filename'], desc='Acoustic test')
])

print('Acoustic train:', X_train.shape)
print('Acoustic test :', X_test.shape)

## 2. Whisper encoder embeddings

In [ ]:
asr = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small.en',
    device=0
)

def get_whisper_embedding(path):
    y, sr = librosa.load(path, sr=16000, mono=True)
    chunk_len = 30 * 16000
    chunks = [y[i:i + chunk_len] for i in range(0, len(y), chunk_len)]

    feats = asr.feature_extractor(
        chunks,
        sampling_rate=16000,
        return_tensors='pt',
        padding='max_length',
        truncation=True
    )

    x = feats.input_features.to('cuda:0')
    with torch.inference_mode():
        h = asr.model.get_encoder()(x).last_hidden_state

    return h.mean(dim=1).mean(dim=0).cpu().numpy().astype(np.float32)

In [ ]:
X_whisper_train = np.vstack([
    get_whisper_embedding(os.path.join(train_audio, fn))
    for fn in tqdm(train['filename'], desc='Whisper train')
])

X_whisper_test = np.vstack([
    get_whisper_embedding(os.path.join(test_audio, fn))
    for fn in tqdm(test['filename'], desc='Whisper test')
])

print('Whisper train:', X_whisper_train.shape)
print('Whisper test :', X_whisper_test.shape)

## 3. Train final ensemble and create submission

In [ ]:
X_tr = np.hstack([X_train, X_whisper_train])
X_te = np.hstack([X_test, X_whisper_test])

print('Combined train:', X_tr.shape)
print('Combined test :', X_te.shape)

cb_final = CatBoostRegressor(
    iterations=1500,
    depth=5,
    learning_rate=0.03,
    loss_function='RMSE',
    l2_leaf_reg=5,
    random_seed=42,
    task_type='GPU',
    devices='0',
    verbose=False
)
cb_final.fit(X_tr, y_train)
pred_cb = cb_final.predict(X_te)

et_final = ExtraTreesRegressor(
    n_estimators=1000,
    max_features=0.9,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
et_final.fit(X_tr, y_train)
pred_et = et_final.predict(X_te)

# Final ensemble used for the submitted solution.
pred = 0.59 * pred_cb + 0.41 * pred_et

# Affine calibration obtained during local validation.
pred_cal = 1.062528 * pred - 0.205908
pred_cal = np.clip(pred_cal, 0, 5)

submission = pd.DataFrame({
    'filename': test['filename'],
    'label': pred_cal
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print('Submission shape:', submission.shape)
print('Missing:', submission.isna().sum().sum())
print(submission.head())
print('Saved: /kaggle/working/submission.csv')